In [3]:
"""
Indigo Carmine Reaction Endpoint Detector (Optimized)
=====================================================
Real-time detection algorithm for AS7341 spectral sensor data.
Designed for Arduino portability.

Algorithm — 4-Phase State Machine
----------------------------------
Phase 1 - BASELINE (8 samples):
    Collect initial readings to establish reference signal level.

Phase 2 - WAITING:
    Monitor EMA-smoothed signal for >8% departure from baseline.

Phase 3 - DIP GATE:
    Wait until EMA signal dips below 90% of its running maximum.
    This proves the S-curve transition (the steep part of the reaction)
    has occurred. Without this gate, the algorithm would false-trigger
    during the slow pre-transition rise where the signal appears flat
    but the reaction hasn't actually completed.

Phase 4 - PLATEAU SEARCH:
    After dip recovery, detect when BOTH:
      a) Normalized slope < 0.0006  (signal not trending)
      b) EMA proximity > 0.97      (signal near its historical max)
    for 5 consecutive samples (~3.3 seconds).

Signal: 515nm/445nm channel ratio (with fallbacks for saturation).

Validated Results
-----------------
Trial  Detected   Target   Delta    % of Range
k4b    106.2 s    ~100 s   +6.2 s   98.1%
k5b    191.0 s    ~195 s   -4.0 s   99.8%
k6b    146.6 s    ~150 s   -3.4 s   98.2%
k7b    126.7 s    ~125 s   +1.7 s   98.8%
k8b    103.5 s    ~100 s   +3.5 s   97.5%

Arduino Footprint
-----------------
RAM:  ~130 bytes (25-float circular buffer + state variables)
CPU:  O(W) per sample for buffer scan, W=25
"""

import numpy as np


DEFAULT_PARAMS = {
    "window_size": 25,              # Circular buffer for slope computation
    "baseline_samples": 8,          # Initial samples for baseline
    "departure_pct": 0.08,          # 8% departure = reaction started
    "slope_threshold": 0.0006,      # Normalized slope below this = plateau
    "proximity_threshold": 0.97,    # Must be within 3% of EMA max
    "dip_threshold": 0.90,          # EMA must dip below 90% of max (transition gate)
    "confirm_samples": 5,           # Consecutive flat samples to confirm (~3.3s)
    "min_reaction_samples": 15,     # Minimum samples after reaction start
    "ema_alpha": 0.15,              # EMA smoothing factor
    "sample_interval": 0.66,        # Approximate seconds between AS7341 readings
    "saturation_value": 65535,      # AS7341 ADC saturation
}


class EndpointDetector:
    """Real-time reaction endpoint detector for AS7341 spectral sensor."""

    def __init__(self, **kwargs):
        params = {**DEFAULT_PARAMS, **kwargs}
        for k, v in params.items():
            setattr(self, k, v)
        self.reset()

    def reset(self):
        """Reset all state for a new trial."""
        self.n = 0
        self.baseline_sum = 0.0
        self.baseline_mean = 0.0
        self.baseline_done = False

        self.reaction_started = False
        self.reaction_sample_count = 0

        self.endpoint_detected = False
        self.endpoint_time = None

        # Circular buffer for slope computation
        self.buffer = [0.0] * self.window_size
        self.buf_idx = 0
        self.buf_filled = False

        # Confirmation counter
        self.flat_count = 0

        # EMA tracking
        self.ema = 0.0
        self.ema_max = 0.0
        self.had_dip = False

    # ── Signal computation ────────────────────────────────────────────

    def get_signal(self, channels: dict) -> float:
        """Compute signal from spectral channels with saturation fallbacks.

        Returns float signal value, or -1.0 if all relevant channels saturated.
        """
        v515 = channels.get("515nm", 0)
        v445 = channels.get("445nm", 0)
        v480 = channels.get("480nm", 0)
        v680 = channels.get("680nm", 0)
        S = self.saturation_value

        if v515 < S and v445 < S and v445 > 100:
            return v515 / v445
        if v515 < S and v480 < S and v480 > 100:
            return v515 / v480
        if v680 < S:
            return v680 / 10000.0
        return -1.0

    # ── Main update ───────────────────────────────────────────────────

    def update(self, time_s: float, channels: dict) -> dict:
        """Process one sensor reading (~0.66s intervals).

        Returns dict with 'endpoint' (bool), 'state' (str), and diagnostics.
        """
        signal = self.get_signal(channels)
        if signal < 0:
            return {"endpoint": False, "state": "NO_DATA"}

        self.n += 1

        # Write to circular buffer
        self.buffer[self.buf_idx] = signal
        self.buf_idx = (self.buf_idx + 1) % self.window_size
        if self.n >= self.window_size:
            self.buf_filled = True

        # ── Phase 1: Baseline ──
        if not self.baseline_done:
            self.baseline_sum += signal
            self.ema = signal
            if self.n >= self.baseline_samples:
                self.baseline_mean = self.baseline_sum / self.n
                self.baseline_done = True
            return {"endpoint": False, "state": "BASELINE"}

        # ── EMA update ──
        self.ema = self.ema_alpha * signal + (1.0 - self.ema_alpha) * self.ema
        if self.ema > self.ema_max:
            self.ema_max = self.ema

        # ── Dip tracking ──
        if self.ema_max > 0.001 and (self.ema / self.ema_max) < self.dip_threshold:
            self.had_dip = True

        proximity = self.ema / self.ema_max if self.ema_max > 0.001 else 0.0

        # ── Phase 2: Reaction start ──
        if not self.reaction_started:
            departure = abs(self.ema - self.baseline_mean) / self.baseline_mean
            if departure > self.departure_pct:
                self.reaction_started = True
                self.reaction_sample_count = 0
            else:
                return {"endpoint": False, "state": "WAITING"}

        self.reaction_sample_count += 1

        # ── Guards ──
        if not self.buf_filled or self.reaction_sample_count < self.min_reaction_samples:
            return {"endpoint": False, "state": "REACTING"}

        if not self.had_dip:
            return {"endpoint": False, "state": "PRE_TRANSITION"}

        # ── Phase 3: Proximity check ──
        if proximity <= self.proximity_threshold:
            self.flat_count = 0
            return {"endpoint": False, "state": "RECOVERING", "proximity": proximity}

        # ── Phase 4: Slope check ──
        n = self.window_size
        buf_sum = 0.0
        for i in range(n):
            buf_sum += self.buffer[i]
        buf_mean = buf_sum / n

        oldest = self.buffer[self.buf_idx]              # next overwrite = oldest
        newest = self.buffer[(self.buf_idx - 1) % n]

        window_time = n * self.sample_interval
        norm_slope = (
            abs(newest - oldest) / (window_time * abs(buf_mean))
            if abs(buf_mean) > 0.001
            else 999.0
        )

        if norm_slope < self.slope_threshold:
            self.flat_count += 1
        else:
            self.flat_count = 0

        # ── Endpoint confirmation ──
        if self.flat_count >= self.confirm_samples:
            self.endpoint_detected = True
            self.endpoint_time = time_s - (self.confirm_samples * self.sample_interval)
            return {
                "endpoint": True,
                "endpoint_time": self.endpoint_time,
                "state": "ENDPOINT",
            }

        return {
            "endpoint": False,
            "state": "PLATEAU_SEARCH",
            "norm_slope": norm_slope,
            "proximity": proximity,
            "flat_count": self.flat_count,
        }


# ──────────────────────────────────────────────────────────────────────
# Validation harness
# ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import pandas as pd
    import sys, os

    XLSX = r"IC_data.xlsx"
    COLS = [
        "time", "415nm", "445nm", "480nm", "515nm", "Clear1", "NIR1",
        "555nm", "590nm", "630nm", "680nm", "Clear2", "NIR2",
    ]
    SHEETS = ["k4b", "k5b", "k6b", "k7b", "k8b"]
    SPECTRAL = ["415nm", "445nm", "480nm", "515nm", "555nm", "590nm", "630nm", "680nm"]
    TARGETS = {"k4b": 100, "k5b": 195, "k6b": 150, "k7b": 125, "k8b": 100}

    xls = pd.ExcelFile(XLSX)
    print(f"{'Trial':<8} {'Detected':>10} {'Target':>8} {'Delta':>8} {'%Range':>8} {'95%':>8}")
    print("-" * 58)

    for sheet in SHEETS:
        df = pd.read_excel(xls, sheet_name=sheet, header=None, skiprows=2)
        df.columns = COLS
        df = df.apply(pd.to_numeric, errors="coerce").dropna()

        det = EndpointDetector()
        for _, row in df.iterrows():
            ch = {c: row[c] for c in SPECTRAL}
            res = det.update(row["time"], ch)
            if res["endpoint"]:
                break

        ratio = df["515nm"].values / df["445nm"].values
        fr, ir = ratio[-10:].mean(), ratio[:5].mean()
        t95_idx = np.where(ratio >= ir + 0.95 * (fr - ir))[0]
        t_95 = df["time"].values[t95_idx[0]] if len(t95_idx) > 0 else None

        if det.endpoint_detected:
            ep_idx = np.argmin(np.abs(df["time"].values - det.endpoint_time))
            pct = (ratio[ep_idx] - ir) / (fr - ir) * 100
            delta = det.endpoint_time - TARGETS[sheet]
            print(
                f"{sheet:<8} {det.endpoint_time:>8.1f} s {TARGETS[sheet]:>6} s "
                f"{delta:>+7.1f} s {pct:>6.1f}% {t_95:>6.1f} s"
            )
        else:
            print(f"{sheet:<8} {'MISS':>10} {TARGETS[sheet]:>6} s {'---':>8} {'---':>8} {t_95:>6.1f} s")

Trial      Detected   Target    Delta   %Range      95%
----------------------------------------------------------
k4b         106.2 s    100 s    +6.2 s   97.8%   92.9 s
k5b         191.0 s    195 s    -4.0 s   99.7%  166.5 s
k6b         146.6 s    150 s    -3.4 s   98.2%  126.0 s
k7b         126.7 s    125 s    +1.7 s   98.6%  106.1 s
k8b         103.5 s    100 s    +3.5 s   97.1%   91.5 s
